# TP : Comptage des mots — Comparaison de trois méthodes

## 1. Introduction

L'objectif de ce TP est de comparer trois approches permettant de compter
le nombre d'occurrences de chaque mot dans un fichier texte.

Les trois méthodes étudiées sont :

1. Python simple : comptage séquentiel à l'aide d'un dictionnaire.
2. Pseudo-MapReduce Python : simulation des étapes Map, Shuffle et Reduce.
3. Hadoop MapReduce : utilisation du framework Hadoop pour effectuer
   le comptage des mots.

Nous allons mesurer les temps d'exécution de ces trois méthodes sur
des fichiers de différentes tailles, vérifier que les résultats obtenus
sont identiques et analyser leurs performances.

## 2. Environnement et outils utilisés

- Système d'exploitation : Ubuntu Linux.
- Langage : Python 3.10.12.
- Environnement Python : environnement virtuel `.venv`.
- Outil de développement : Jupyter Notebook.
- Framework distribué prévu : Apache Hadoop.

## 3. Génération du fichier de test

Nous créons un fichier texte nommé `mots.txt` contenant un mot par ligne.

Le fichier contient volontairement des répétitions afin de pouvoir
mesurer le nombre d'occurrences de chaque mot.

Ce fichier servira de référence pour tester les trois méthodes.

In [1]:
from pathlib import Path
from collections import Counter

# Définition du fichier de test
fichier_mots = Path("mots.txt")

# Liste de mots avec des répétitions
mots = [
    "chat", "chien", "chat", "voiture", "chat",
    "arbre", "chien", "fleur", "chat", "voiture",
    "soleil", "arbre", "chat", "chien", "fleur",
    "livre", "chat", "montagne", "chien", "soleil"
]

# Création du fichier : un mot par ligne
with open(fichier_mots, "w", encoding="utf-8") as f:
    for mot in mots:
        f.write(mot + "\n")

print("Fichier créé :", fichier_mots.resolve())
print("Nombre de lignes :", len(mots))
print("Contenu du fichier :")
print(fichier_mots.read_text(encoding="utf-8"))

Fichier créé : /home/ssl/Documents/5A/S9/BD/TP_Comptage_Mots/mots.txt
Nombre de lignes : 20
Contenu du fichier :
chat
chien
chat
voiture
chat
arbre
chien
fleur
chat
voiture
soleil
arbre
chat
chien
fleur
livre
chat
montagne
chien
soleil



## 4. Méthode 1 : Python simple

### Principe

La première méthode utilise un dictionnaire Python pour compter
les occurrences de chaque mot dans le fichier.

Le programme lit le fichier, récupère tous les mots avec la méthode
`split()`, puis parcourt les mots un par un.

Pour chaque mot, le compteur correspondant est incrémenté de 1.
Les résultats sont ensuite triés par ordre décroissant du nombre
d'occurrences.

Cette méthode est séquentielle : les mots sont traités les uns
après les autres, sans parallélisation.

In [2]:
def compter_occurrences(fichier):
    """
    Compte les occurrences de chaque mot dans un fichier texte.

    Paramètre :
        fichier : chemin du fichier à analyser.

    Retour :
        Un dictionnaire contenant chaque mot et son nombre d'occurrences.
    """
    with open(fichier, "r", encoding="utf-8") as f:
        mots = f.read().split()

    compteur = {}

    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1

    return compteur


# Exécution de la méthode 1
resultats_python = compter_occurrences(fichier_mots)

# Affichage des résultats par ordre décroissant
for mot, nombre in sorted(
    resultats_python.items(),
    key=lambda element: element[1],
    reverse=True
):
    print(f"{mot} : {nombre}")

chat : 6
chien : 4
voiture : 2
arbre : 2
fleur : 2
soleil : 2
livre : 1
montagne : 1


In [3]:
# Vérification du nombre total de mots comptabilisés
total_mots = sum(resultats_python.values())

print("Nombre total de mots :", total_mots)
print("Nombre de mots différents :", len(resultats_python))

assert total_mots == 20, "Erreur : le total doit être égal à 20."

print("Vérification réussie : les 20 mots ont été comptabilisés.")

Nombre total de mots : 20
Nombre de mots différents : 8
Vérification réussie : les 20 mots ont été comptabilisés.


### Mesure du temps d'exécution

Nous utilisons le module `time` pour mesurer le temps nécessaire
à l'exécution de la méthode Python simple.

La mesure inclut la lecture du fichier et le comptage des mots.

In [4]:
import time

debut = time.perf_counter()

resultats_python = compter_occurrences(fichier_mots)

fin = time.perf_counter()

temps_python = fin - debut

print(f"Temps d'exécution : {temps_python:.6f} seconde(s)")

Temps d'exécution : 0.000340 seconde(s)


## 5. Méthode 2 : Pseudo-MapReduce Python

### Principe

Cette deuxième méthode reproduit localement les trois étapes du modèle
MapReduce : Map, Shuffle et Reduce.

- Map : chaque mot est transformé en une paire (mot, 1).
- Shuffle : les valeurs associées à chaque mot sont regroupées.
- Reduce : les valeurs de chaque groupe sont additionnées pour obtenir
  le nombre total d'occurrences.

Contrairement à Hadoop, cette implémentation s'exécute entièrement
sur une seule machine. Elle permet néanmoins de comprendre
l'organisation du traitement MapReduce.

In [5]:
from collections import defaultdict


def mapper(ligne):
    """
    Transforme chaque mot d'une ligne en une paire (mot, 1).
    """
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]


def shuffle(mapped_data):
    """
    Regroupe les valeurs associées à chaque mot.
    """
    grouped = defaultdict(list)

    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)

    return grouped


def reducer(grouped_data):
    """
    Additionne les valeurs associées à chaque mot.
    """
    reduced = {}

    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)

    return reduced


# Exécution de la méthode 2
mapped = []

with open(fichier_mots, "r", encoding="utf-8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))

grouped = shuffle(mapped)
resultats_mapreduce = reducer(grouped)

# Affichage des résultats
for mot, nombre in sorted(
    resultats_mapreduce.items(),
    key=lambda element: element[1],
    reverse=True
):
    print(f"{mot} : {nombre}")

chat : 6
chien : 4
voiture : 2
arbre : 2
fleur : 2
soleil : 2
livre : 1
montagne : 1


In [6]:
# Comparaison des résultats des deux méthodes
assert resultats_python == resultats_mapreduce, (
    "Erreur : les deux méthodes donnent des résultats différents."
)

print("Vérification réussie : les deux méthodes donnent les mêmes résultats.")

print("Total des mots :", sum(resultats_mapreduce.values()))
print("Nombre de mots différents :", len(resultats_mapreduce))

Vérification réussie : les deux méthodes donnent les mêmes résultats.
Total des mots : 20
Nombre de mots différents : 8


In [7]:
debut = time.perf_counter()

mapped = []

with open(fichier_mots, "r", encoding="utf-8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))

grouped = shuffle(mapped)
resultats_mapreduce = reducer(grouped)

fin = time.perf_counter()

temps_mapreduce = fin - debut

print(f"Temps d'exécution : {temps_mapreduce:.6f} seconde(s)")

Temps d'exécution : 0.000383 seconde(s)


## 6. Comparaison des performances sur différentes tailles de fichiers

### Objectif

Nous cherchons à comparer les performances des deux implémentations
Python en faisant varier la taille des fichiers d'entrée.

Nous générons plusieurs fichiers en dupliquant le contenu du fichier
initial `mots.txt`. Chaque ligne contient un mot.

Pour chaque fichier, nous exécutons les deux méthodes :
- Python simple ;
- Pseudo-MapReduce Python.

Nous mesurons leur temps d'exécution et vérifions que les résultats
obtenus sont identiques.

Les temps sont mesurés plusieurs fois et nous retenons la médiane
afin de limiter l'influence des variations ponctuelles du système.

In [1]:
from pathlib import Path

# Fichier de référence contenant les 20 mots
fichier_source = Path("mots.txt")

# Dossier des fichiers de test
dossier_tests = Path("fichiers_test")
dossier_tests.mkdir(exist_ok=True)

# Facteurs de duplication
facteurs = [1, 10, 100, 1000, 5000]

# Lecture du fichier initial
contenu = fichier_source.read_text(encoding="utf-8")

# Génération des fichiers
fichiers_generes = {}

for m in facteurs:
    fichier_test = dossier_tests / f"mots_{m}.txt"

    with open(fichier_test, "w", encoding="utf-8") as f:
        f.write(contenu * m)

    fichiers_generes[m] = fichier_test

    print(
        f"Facteur m = {m:5d} | "
        f"Lignes = {20 * m:7d} | "
        f"Taille = {fichier_test.stat().st_size / 1024:.2f} Ko"
    )

Facteur m =     1 | Lignes =      20 | Taille = 0.12 Ko
Facteur m =    10 | Lignes =     200 | Taille = 1.20 Ko
Facteur m =   100 | Lignes =    2000 | Taille = 12.01 Ko
Facteur m =  1000 | Lignes =   20000 | Taille = 120.12 Ko
Facteur m =  5000 | Lignes =  100000 | Taille = 600.59 Ko


In [4]:
import time
import statistics
import pandas as pd


def compter_python(fichier):
    """Méthode 1 : comptage séquentiel avec un dictionnaire."""
    with open(fichier, "r", encoding="utf-8") as f:
        mots = f.read().split()

    compteur = {}

    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1

    return compteur


def compter_pseudo_mapreduce(fichier):
    """Méthode 2 : Map, Shuffle et Reduce."""
    mapped = []

    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))

    grouped = shuffle(mapped)

    return reducer(grouped)


def mesurer_temps(fonction, fichier, repetitions=5):
    """Mesure le temps médian d'exécution."""
    durees = []

    for _ in range(repetitions):
        debut = time.perf_counter()
        fonction(fichier)
        fin = time.perf_counter()

        durees.append(fin - debut)

    return statistics.median(durees)


resultats_benchmarks = []

for m, fichier in fichiers_generes.items():

    # Mesure des deux méthodes
    temps_simple = mesurer_temps(compter_python, fichier)
    temps_pseudo = mesurer_temps(compter_pseudo_mapreduce, fichier)

    # Vérification des résultats
    resultat_simple = compter_python(fichier)
    resultat_pseudo = compter_pseudo_mapreduce(fichier)

    assert resultat_simple == resultat_pseudo, (
        f"Résultats différents pour le facteur m = {m}"
    )

    resultats_benchmarks.append({
        "Facteur m": m,
        "Nombre de lignes": 20 * m,
        "Taille (Ko)": round(fichier.stat().st_size / 1024, 2),
        "Python simple (s)": temps_simple,
        "Pseudo-MapReduce (s)": temps_pseudo
    })

df_benchmarks = pd.DataFrame(resultats_benchmarks)

print("Tous les tests de comptage sont réussis.")
display(df_benchmarks)

NameError: name 'mapper' is not defined

In [3]:
%pip install pandas

  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]━━━━━━━━━━━ 2/3 [pandas]
Note: you may need to restart the kernel to use updated packages.
